# Arithmetic Transformer Runner

Single notebook for local computer or Google Colab execution.

In [ ]:
# Set RUN_ENV to 'local' on your computer or 'colab' in Google Colab.
RUN_ENV = 'colab'

GIT_REPO_URL = 'https://github.com/ffbskt/AgentAI.git'
LOCAL_REPO_DIR = r'D:/Codex projects/Transformer_Graph3/arithmetic-transformer'
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/AgentAI'

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if RUN_ENV == 'colab':
    project_root = Path(COLAB_PROJECT_ROOT)
    repo_root = project_root
    git_dir = repo_root / '.git'
    if git_dir.exists():
        print(f"'{repo_root}' is already a git repository. Pulling latest changes...")
        subprocess.run(['git', '-C', str(repo_root), 'pull'], check=True)
    elif repo_root.exists():
        print(f"'{repo_root}' exists but is not a git repository. Removing and re-cloning...")
        shutil.rmtree(repo_root)
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning '{GIT_REPO_URL}' into '{repo_root}'...")
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)
    REPO_DIR = repo_root / 'arithmetic-transformer'
else:
    REPO_DIR = Path(LOCAL_REPO_DIR)

CHECKPOINT_DIR = REPO_DIR / 'checkpoints'
RUNS_DIR = REPO_DIR / 'runs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo dir:', REPO_DIR)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Runs dir:', RUNS_DIR)

In [ ]:
%cd {REPO_DIR}
!python -m pip install -q -r requirements-colab.txt

In [ ]:
# Change DEVICE to 'cuda' in Colab GPU runtime if available.
DEVICE = 'cpu'
MODEL_KIND = 'lstm'
DROPOUT = '0.01'
EXTRA_ARGS = ''

cmd = f'''python train.py \
  --kind {MODEL_KIND} \
  --device {DEVICE} \
  --dropout {DROPOUT} \
  --save-dir "{CHECKPOINT_DIR}" \
  --save-each-digit \
  --protocol-file "{RUNS_DIR / 'protocol.jsonl'}" \
  {EXTRA_ARGS} \
  | tee "{RUNS_DIR / 'train.log'}"'''

print(cmd)

In [ ]:
!{cmd}

In [ ]:
!ls -lah "$CHECKPOINT_DIR"
!tail -n 20 "$RUNS_DIR/protocol.jsonl" || true